# Global Window, Glazing & Glass Website Harvester (100k–500k+ URLs)

Collect **hundreds of thousands of deduplicated business website URLs** across every single window vertical — **double/triple/secondary glazing, uPVC, timber & sash, aluminium & bi-fold, steel & Crittall, window manufacturers, trade suppliers, glaziers, flat/toughened/sealed-unit glass, skylights, roof lanterns, conservatories, and commercial curtain walling**.

### How to run in Google Colab
1. Click **Runtime → Run all** (no API keys required!).
2. Approve the Google Drive mount prompt so your database and `urls.txt` save permanently to `MyDrive/window_websites/` (if Colab disconnects, re-running picks up from the exact checkpoint — zero progress lost!).
3. **Free zero-key mode (`free-fast` / `free-max`)** pulls **150,000–350,000+ real URLs** using Overture Maps (60M+ global POIs), Common Crawl (100M+ active domains + service-page CDX index), OpenStreetMap (50 countries), Certificate Transparency SSL logs (`crt.sh`), DuckDuckGo + Brave + Startpage (Google) + Yahoo (Bing) + Mojeek, and YellowPages.
4. **Want 400,000–600,000+ URLs even faster?** Paste an optional `SERPER_API_KEY` (Google Maps + Google Search), `GOOGLE_PLACES_API_KEY` (Adaptive Quadtree Google Maps), `BRAVE_API_KEY`, or free `COMPANIES_HOUSE_API_KEY` in Step 2.

In [ ]:
#@title Step 1 — Mount Google Drive & Install Window Harvester
import os
import subprocess
import sys
from pathlib import Path

SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER_NAME = "window_websites"  #@param {type:"string"}
REPO_URL = "https://github.com/FredTheFork/emailscraper.git"  #@param {type:"string"}

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
else:
    WORK_ROOT = Path.cwd() / DRIVE_FOLDER_NAME

DB_PATH = WORK_ROOT / "data" / "window_websites.db"
OUTPUT_DIR = WORK_ROOT / "output"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If running outside the cloned repo (e.g. fresh Colab VM), clone & install
if not Path("window_harvester").is_dir():
    repo_dir = Path("/content/emailscraper") if IN_COLAB else Path("./emailscraper")
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_dir)])
    else:
        subprocess.call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    if not (repo_dir / "window_harvester").is_dir():
        subprocess.call(["git", "-C", str(repo_dir), "fetch", "origin", "arena/07954137-emailscraper"])
        subprocess.call(["git", "-C", str(repo_dir), "checkout", "arena/07954137-emailscraper"])
    os.chdir(repo_dir)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print(f"Ready! Database: {DB_PATH}")
print(f"Exports folder:  {OUTPUT_DIR}")

In [ ]:
#@title Step 2 — Choose Harvest Preset, Scope & Optional API Keys
#@markdown ### Harvest Strategy
PRESET = "free-max"  #@param ["free-fast", "free-max", "uk-complete", "us-complete", "english-world", "maps-blitz", "global-500k"]
COUNTRIES_OVERRIDE = ""  #@param {type:"string"}
#@markdown *(Leave `COUNTRIES_OVERRIDE` blank to use the preset default, or set e.g. `GB` or `GB,US,CA,AU,IE,NZ`)*
TARGET_DOMAINS = 350000  #@param {type:"integer"}
CHUNK_SIZE = 50000  #@param {type:"integer"}
SPLIT_BY_COUNTRY = True  #@param {type:"boolean"}
SPLIT_BY_CATEGORY = True  #@param {type:"boolean"}

#@markdown ### Optional Sub-Vertical Toggles
#@markdown *(Double glazing, uPVC, timber, sash, aluminium, steel/Crittall, suppliers, manufacturers, glaziers, glass, skylights & conservatories are ALWAYS included)*
INCLUDE_TREATMENTS = False  #@param {type:"boolean"}
INCLUDE_WINDOW_CLEANING = False  #@param {type:"boolean"}
INCLUDE_AUTO_GLASS = False  #@param {type:"boolean"}

#@markdown ### Optional API Keys (leave blank for 100% free mode, or fill any to turbocharge Google Maps / Google / Brave)
SERPER_API_KEY = ""  #@param {type:"string"}
GOOGLE_PLACES_API_KEY = ""  #@param {type:"string"}
BRAVE_API_KEY = ""  #@param {type:"string"}
SERPAPI_API_KEY = ""  #@param {type:"string"}
DATAFORSEO_LOGIN = ""  #@param {type:"string"}
DATAFORSEO_PASSWORD = ""  #@param {type:"string"}
COMPANIES_HOUSE_API_KEY = ""  #@param {type:"string"}
PROXIES_CSV = ""  #@param {type:"string"}

def _get_secret(name: str, fallback: str) -> str:
    if fallback.strip():
        return fallback.strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val.strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()

from window_harvester.config import PRESETS, HarvestConfig
from window_harvester.models import FilterOptions

spec = PRESETS[PRESET]
explicit_ccs = [c.strip().upper() for c in COUNTRIES_OVERRIDE.split(",") if c.strip()] or None
effective_ccs = explicit_ccs if explicit_ccs is not None else (list(spec.countries) if spec.countries else None)

cfg = HarvestConfig(
    db_path=DB_PATH,
    output_dir=OUTPUT_DIR,
    countries=effective_ccs,
    search_countries=explicit_ccs or list(spec.search_countries),
    keyword_tier=spec.keyword_tier,
    geo_tier=spec.geo_tier,
    include_uk_postcodes=spec.include_uk_postcodes,
    target_domains=TARGET_DOMAINS if TARGET_DOMAINS > 0 else None,
    serper_api_key=_get_secret("SERPER_API_KEY", SERPER_API_KEY),
    google_places_api_key=_get_secret("GOOGLE_PLACES_API_KEY", GOOGLE_PLACES_API_KEY),
    brave_api_key=_get_secret("BRAVE_API_KEY", BRAVE_API_KEY),
    serpapi_api_key=_get_secret("SERPAPI_API_KEY", SERPAPI_API_KEY),
    dataforseo_login=_get_secret("DATAFORSEO_LOGIN", DATAFORSEO_LOGIN),
    dataforseo_password=_get_secret("DATAFORSEO_PASSWORD", DATAFORSEO_PASSWORD),
    companies_house_api_key=_get_secret("COMPANIES_HOUSE_API_KEY", COMPANIES_HOUSE_API_KEY),
    proxies=[p.strip() for p in PROXIES_CSV.split(",") if p.strip()],
    filters=FilterOptions(
        min_confidence=0.55,
        include_treatments=INCLUDE_TREATMENTS,
        include_window_cleaning=INCLUDE_WINDOW_CLEANING,
        include_auto_glass=INCLUDE_AUTO_GLASS,
        allowed_countries=set(effective_ccs) if effective_ccs else None,
    ),
)

sources_to_run = cfg.resolve_preset_sources(PRESET)
# Also attach any API keys you pasted even when running free-fast / free-max
for extra_api in cfg.configured_api_sources():
    if extra_api not in sources_to_run:
        sources_to_run.append(extra_api)

print(f"Preset:          {PRESET} ({spec.estimated_yield})")
print(f"Country scope:   {effective_ccs or 'GLOBAL (all countries)'}")
print(f"Active sources:  {', '.join(sources_to_run)}")

In [ ]:
#@title Step 3 — Run the Harvester (Auto-Checkpoints to Google Drive!)
#@markdown You can stop and re-run this cell anytime — completed tasks are skipped automatically and all URLs stay deduplicated in SQLite.
from window_harvester.pipeline import run_harvest_pipeline

run_results = run_harvest_pipeline(
    cfg,
    sources_to_run,
    verify_after=False,
    export_after=True,
    chunk_size=CHUNK_SIZE if CHUNK_SIZE > 0 else None,
    split_by_country=SPLIT_BY_COUNTRY,
    split_by_category=SPLIT_BY_CATEGORY,
)

In [ ]:
#@title Step 4 (Optional) — Direct Free Google Maps Browser Scraper (No API Key)
#@markdown Flip `RUN_FREE_GMAPS_BROWSER = True` to also scroll Google Maps listings directly in headless Chromium across UK/US/global cities & postcodes.
RUN_FREE_GMAPS_BROWSER = False  #@param {type:"boolean"}
MAX_GMAPS_QUERIES_THIS_SESSION = 500  #@param {type:"integer"}

if RUN_FREE_GMAPS_BROWSER:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright"])
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"])
    cfg.max_tasks_per_source = MAX_GMAPS_QUERIES_THIS_SESSION
    run_harvest_pipeline(
        cfg,
        ["playwright_gmaps"],
        export_after=True,
        chunk_size=CHUNK_SIZE if CHUNK_SIZE > 0 else None,
        split_by_country=SPLIT_BY_COUNTRY,
        split_by_category=SPLIT_BY_CATEGORY,
    )
    cfg.max_tasks_per_source = None
else:
    print("Skipped optional Playwright Google Maps pass (set RUN_FREE_GMAPS_BROWSER = True to run).")

In [ ]:
#@title Step 5 (Optional) — High-Speed Async Live Website Verifier
#@markdown Checks domains at 150 concurrent requests/sec to prune dead domains, parked-for-sale pages, and any remaining non-window sites, then re-exports verified lists.
RUN_LIVE_VERIFIER = False  #@param {type:"boolean"}
VERIFY_LIMIT = 50000  #@param {type:"integer"}
VERIFY_CONCURRENCY = 150  #@param {type:"integer"}

if RUN_LIVE_VERIFIER:
    from window_harvester.db import HarvestDB
    from window_harvester.exporter import export_websites
    from window_harvester.verifier import verify_database_async

    with HarvestDB(DB_PATH) as db:
        summary = await verify_database_async(
            db,
            options=cfg.filters,
            concurrency=VERIFY_CONCURRENCY,
            limit=VERIFY_LIMIT,
            on_progress=lambda s: print(
                f"Checked {s.checked:,} | Live window sites: {s.live_relevant:,} | "
                f"Dead: {s.dead_or_error:,} | Parked: {s.parked:,} | Filtered: {s.irrelevant:,}"
            ),
        )
        exp = export_websites(
            db,
            OUTPUT_DIR,
            fmt="all",
            min_confidence=cfg.filters.min_confidence,
            chunk_size=CHUNK_SIZE if CHUNK_SIZE > 0 else None,
            split_by_country=SPLIT_BY_COUNTRY,
            split_by_category=SPLIT_BY_CATEGORY,
        )
        print(f"Re-exported {exp['exported_count']:,} clean window business URLs to {OUTPUT_DIR}")
else:
    print("Skipped optional live verification pass (set RUN_LIVE_VERIFIER = True to run).")

In [ ]:
#@title Step 6 — Preview Collected URLs & Download `urls.txt` / `websites.csv`
DOWNLOAD_URLS_TXT = False  #@param {type:"boolean"}
DOWNLOAD_WEBSITES_CSV = False  #@param {type:"boolean"}

urls_file = OUTPUT_DIR / "urls.txt"
csv_file = OUTPUT_DIR / "websites.csv"

if urls_file.exists():
    with urls_file.open("r", encoding="utf-8") as fh:
        sample = [next(fh, "").strip() for _ in range(25)]
    total_lines = sum(1 for _ in urls_file.open("r", encoding="utf-8"))
    print(f"Total unique window website URLs in {urls_file}: {total_lines:,}\n")
    print("First 25 URLs:")
    for u in [s for s in sample if s]:
        print(f"  {u}")

    if IN_COLAB:
        from google.colab import files
        if DOWNLOAD_URLS_TXT:
            files.download(str(urls_file))
        if DOWNLOAD_WEBSITES_CSV and csv_file.exists():
            files.download(str(csv_file))
else:
    print("Run Step 3 first to generate urls.txt and websites.csv.")